In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 34.1 MB/s eta 0:00:00


In [ ]:
import os
import cv2
import numpy as np
from ultralytics import YOLO

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
yolo = YOLO("yolov8n.pt")

In [ ]:
def crop_buffalo(img_path):
    results = yolo(img_path)
    img = cv2.imread(img_path)

    for r in results:
        boxes = r.boxes.xyxy.cpu().numpy()
        if len(boxes) > 0:
            x1, y1, x2, y2 = map(int, boxes[0])
            return img[y1:y2, x1:x2]

    return img

In [ ]:
input_base = "/content/drive/MyDrive/Animal dataset/Buffalo dataset"
output_base = "/content/drive/MyDrive/Animal dataset/Buffalo cropped"

for split in ["train", "test"]:
    for cls in os.listdir(os.path.join(input_base, split)):

        in_path = os.path.join(input_base, split, cls)
        out_path = os.path.join(output_base, split, cls)

        os.makedirs(out_path, exist_ok=True)

        for img_name in os.listdir(in_path):
            img_path = os.path.join(in_path, img_name)

            crop = crop_buffalo(img_path)

            if crop is not None:
                crop = cv2.resize(crop, (224,224))
                cv2.imwrite(os.path.join(out_path, img_name), crop)


image 1/1 /content/drive/MyDrive/Animal dataset/Buffalo dataset/train/Nagpuri/image_259x194_6 (2).jpg: 640x640 1 cow, 7.7ms
Speed: 10.6ms preprocess, 7.7ms inference, 34.6ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 /content/drive/MyDrive/Animal dataset/Buffalo dataset/train/Nagpuri/image_330x144_69.jpg: 640x640 1 horse, 34.3ms
Speed: 3.2ms preprocess, 34.3ms inference, 2.2ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 /content/drive/MyDrive/Animal dataset/Buffalo dataset/train/Nagpuri/image_259x194_7.jpg: 640x640 1 elephant, 11.0ms
Speed: 2.5ms preprocess, 11.0ms inference, 1.3ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 /content/drive/MyDrive/Animal dataset/Buffalo dataset/train/Nagpuri/FileErumajpg_-_Wikimedia_Commons.jpg: 640x640 (no detections), 12.8ms
Speed: 2.7ms preprocess, 12.8ms inference, 1.4ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 /content/drive/MyDrive/Animal dataset/Buffalo dataset/train/Nagpuri/Cattle

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    zoom_range=0.15,
    horizontal_flip=True
)

test_gen = ImageDataGenerator(rescale=1./255)

In [ ]:
train_path = os.path.join(output_base, "train")
test_path = os.path.join(output_base, "test")

train_data = train_gen.flow_from_directory(
    train_path,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical'
)

test_data = test_gen.flow_from_directory(
    test_path,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical'
)

Found 492 images belonging to 4 classes.
Found 103 images belonging to 4 classes.


In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

In [ ]:
num_classes = train_data.num_classes
print("Classes:", num_classes)

Classes: 4


In [ ]:
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
for layer in base_model.layers:
    layer.trainable = False

In [ ]:
x = base_model.output
x = GlobalAveragePooling2D()(x)

x = BatchNormalization()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)

x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)

output = Dense(num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    train_data,
    validation_data=test_data,
    epochs=30
)

Epoch 1/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 13s 800ms/step - accuracy: 0.7764 - loss: 0.5801 - val_accuracy: 0.5340 - val_loss: 1.3069
Epoch 2/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 9s 579ms/step - accuracy: 0.7683 - loss: 0.5940 - val_accuracy: 0.5340 - val_loss: 1.3105
Epoch 3/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 7s 433ms/step - accuracy: 0.7541 - loss: 0.6074 - val_accuracy: 0.5340 - val_loss: 1.3016
Epoch 4/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 9s 552ms/step - accuracy: 0.8008 - loss: 0.5346 - val_accuracy: 0.5340 - val_loss: 1.3135
Epoch 5/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 7s 442ms/step - accuracy: 0.7683 - loss: 0.5762 - val_accuracy: 0.5340 - val_loss: 1.3304
Epoch 6/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 9s 529ms/step - accuracy: 0.7602 - loss: 0.6011 - val_accuracy: 0.5340 - val_loss: 1.3326
Epoch 7/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 8s 486ms/step - accuracy: 0.8232 - loss: 0.4895 - val_accuracy: 0.5340 - val_loss: 1.3263
Epoch 8/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 8s 500ms/step - accuracy: 0.8232 - loss: 0.4787 - val_accuracy: 0

In [ ]:
loss, acc = model.evaluate(test_data)
print("Final Accuracy:", acc)

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step - accuracy: 0.5340 - loss: 1.4174 
Final Accuracy: 0.5339806079864502


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving image_45.jpg to image_45.jpg


In [ ]:
img_name = list(uploaded.keys())[0]

def final_predict(img_path):
    cropped_img = crop_buffalo(img_path)
    if cropped_img is None:
        return {"error": "No buffalo detected in the image."}

    # Preprocess the cropped image for the MobileNetV2 model
    img_array = cv2.resize(cropped_img, (224, 224))
    img_array = np.expand_dims(img_array, axis=0) # Add batch dimension
    img_array = img_array / 255.0 # Rescale to [0, 1]

    predictions = model.predict(img_array)
    predicted_class_index = np.argmax(predictions, axis=1)[0]
    predicted_probability = predictions[0][predicted_class_index]

    # Get class labels from train_data
    class_labels = {v: k for k, v in train_data.class_indices.items()}
    predicted_class_name = class_labels[predicted_class_index]

    return {"class": predicted_class_name, "probability": float(predicted_probability)}


result = final_predict(img_name)

for k,v in result.items():
    print(k, ":", v)


image 1/1 /content/image_45.jpg: 448x640 1 cow, 7.6ms
Speed: 2.6ms preprocess, 7.6ms inference, 1.3ms postprocess per image at shape (1, 3, 448, 640)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
class : Surti
probability : 0.8369105458259583


In [ ]:
model.save("/content/drive/MyDrive/buffalo_model.h5")

In [ ]:
import os
print(os.listdir("/content/drive/MyDrive"))

['Colab Notebooks', 'appar id.pdf', 'Classroom', 'DSP', 'Temp.pdf_copy.pdf', 'Final Synopsis Format  P1 (1) (1).gdoc', 'Final Synopsis Format  P1 (1).gdoc', 'insertatbeg_fc63fa7f7534f4491a6947c3986fa2e3 (1).cpp.gdoc', 'insatend_081f63c6131310df45b4a8688e0a6a0b (1).cpp.gdoc', 'insertatbeg_fc63fa7f7534f4491a6947c3986fa2e3.cpp.gdoc', 'insatend_081f63c6131310df45b4a8688e0a6a0b.cpp.gdoc', 'instat specific_0bbde1a9dd1d7878f024071ed22199ae.cpp.gdoc', 'delete_0a3b8b6ff4d2291b5e07c98e0b7d713e (3).cpp.gdoc', 'delete_0a3b8b6ff4d2291b5e07c98e0b7d713e (2).cpp.gdoc', 'delete_0a3b8b6ff4d2291b5e07c98e0b7d713e (1).cpp.gdoc', 'delete_0a3b8b6ff4d2291b5e07c98e0b7d713e.cpp.gdoc', 'binary tree_ad469c537aa5b6f69551495b24954f16 (1).cpp.gdoc', 'BST_42d3ae18b8feff5cc8c9cfb24a7dce81 (1).cpp.gdoc', 'binary tree_ad469c537aa5b6f69551495b24954f16.cpp.gdoc', 'BST_42d3ae18b8feff5cc8c9cfb24a7dce81.cpp.gdoc', 'insertion all 3_309d77ab5bf71bd09b3318663f865286.cpp.gdoc', '2403153 (1).pdf', '2403153 Nasscom cert.pdf', 'Nas